In [1]:
import torch
import torchaudio
import numpy as np
import os

# 1. Cấu hình các thông số giống lúc train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 66
audio_path_to_test = r'D:\voeicNCKH\Processed_Audio_5s\Achetadomesticus_XC489192-Achetadomesticus_poland_psz_20140510_22.00h_3498_edit1_seg0.wav' # Thay bằng tên file thực tế của bạn

# 2. Nạp từ điển tên loài (Nếu bạn đã lưu file classes.npy)
classes = np.load('classes.npy', allow_pickle=True)

# 3. Nạp lại kiến trúc Model
from torchvision import models
import torch.nn as nn

def load_saved_model(path, num_classes):
    model = models.resnet18()
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(torch.load(path))
    model.to(device)
    model.eval()
    return model

# 4. Hàm xử lý âm thanh đầu vào
def predict_insect(model, audio_path):
    # Load và biến đổi giống hệt lúc train
    waveform, sr = torchaudio.load(audio_path)
    
    # Mel Spectrogram
    mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=22050, n_mels=128).to(device)
    db_transform = torchaudio.transforms.AmplitudeToDB().to(device)
    
    spec = mel_transform(waveform.to(device))
    spec = db_transform(spec)
    
    # Thêm chiều batch (1, 1, H, W)
    spec = spec.unsqueeze(0) 
    
    with torch.no_grad():
        outputs = model(spec)
        _, predicted = torch.max(outputs, 1)
        species_name = classes[predicted.item()]
    
    return species_name

# --- CHẠY THỬ ---
model = load_saved_model('insect_detector.pth', num_classes)
result = predict_insect(model, audio_path_to_test)

print(f"Kết quả dự đoán: {result}")


C:\Users\luan0\AppData\Local\Temp\ipykernel_6684\1349564100.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path))
d:\voeicNCKH\venv\li

Kết quả dự đoán: Achetadomesticus


In [2]:
# 1. Bảng tra cứu Tần số đuổi (Repel Database)
# Lưu ý: Các con số này là ví dụ, bạn có thể điều chỉnh theo tài liệu sinh học
REPEL_DB = {
    'Achetadomesticus': 22000,   # Dế nhà sợ sóng 22kHz
    'Tettigoniacantans': 19000,
    'Yoyettacelis': 25000,
    # Thêm các loài khác...
}

def trigger_repel_action(species_name):
    print(f"\n[HỆ THỐNG PHẢN PHÁT]")
    print(f"Đang xử lý loài: {species_name}")
    
    # Lấy tần số từ DB, nếu không có thì mặc định là 20kHz (siêu âm)
    freq = REPEL_DB.get(species_name, 20000)
    
    print(f">>> HÀNH ĐỘNG: Đang phát sóng âm {freq} Hz để đuổi {species_name}...")
    print(f">>> TRẠNG THÁI: Đang bảo vệ khu vực.")

# 2. Chạy thử quy trình hoàn chỉnh
predicted_species = result # Đây là kết quả 'Achetadomesticus' bạn vừa nhận được
trigger_repel_action(predicted_species)



[HỆ THỐNG PHẢN PHÁT]
Đang xử lý loài: Achetadomesticus
>>> HÀNH ĐỘNG: Đang phát sóng âm 22000 Hz để đuổi Achetadomesticus...
>>> TRẠNG THÁI: Đang bảo vệ khu vực.
